# Baseline Evaluation for comparison

Install dependencies

In [1]:
import importlib.util

REQUIRED_PACKAGES = ["transformers", "torch", "sklearn", "joblib", "pandas", "numpy"]
MISSING_PACKAGES = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if MISSING_PACKAGES:
    print("Installing missing packages:", MISSING_PACKAGES)
    !pip install -q transformers torch scikit-learn joblib pandas numpy
else:
    print("All required packages already available.")


All required packages already available.


# Imports and use cuda

In [2]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


# Mount the google drive containing data and models

In [3]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


# Set paths and Config

In [4]:

BASE_DIR = "/content/drive/MyDrive/Capstone"

DATA_DIR = os.path.join(BASE_DIR, "data_v2")
SAVED_DIR = os.path.join(BASE_DIR, "saved")

TRADITIONAL_DIR = os.path.join(SAVED_DIR, "traditional_ml_v2")
DISTILBERT_DIR = os.path.join(SAVED_DIR, "distilbert", "distilbert_best")
MODERNBERT_DIR = os.path.join(SAVED_DIR, "modernbert", "modernbert_best")

TEST_CSV_PATH = os.path.join(DATA_DIR, "test_v2.csv")

EVAL_RESULTS_DIR = os.path.join(BASE_DIR, "eval", "results")
os.makedirs(EVAL_RESULTS_DIR, exist_ok=True)


DECISION_THRESHOLD = 0.5


DISTILBERT_MAX_LEN = 512
DISTILBERT_EVAL_BATCH_SIZE = 32

MODERNBERT_MAX_LEN = 4096
MODERNBERT_EVAL_BATCH_SIZE = 8  # kept small: max_len=4096 is memory-heavy on a T4

print("BASE_DIR:", BASE_DIR)
print("TEST_CSV_PATH:", TEST_CSV_PATH)


BASE_DIR: /content/drive/MyDrive/Capstone
TEST_CSV_PATH: /content/drive/MyDrive/Capstone/data_v2/test_v2.csv


# Load test dataset

In [5]:
test_df = pd.read_csv(TEST_CSV_PATH)

# Arrow-string-dtype / NaN quirk from earlier pipeline work 
test_df["text"] = test_df["text"].fillna("").astype(str)

y_true = test_df["label"].to_numpy()
texts = test_df["text"].tolist()

print(f"Test set size: {len(test_df)}")
print(test_df["label"].value_counts())


Test set size: 3093
label
0    1613
1    1480
Name: count, dtype: int64


# Helper functions (mostly copied from earlier work with few modifications)

In [6]:
def get_sklearn_scores(model, features):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    return model.decision_function(features)


def evaluate_predictions(y_true_labels, y_pred_labels, y_scores, model_name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_labels, y_pred_labels, average="binary", pos_label=1
    )
    accuracy = accuracy_score(y_true_labels, y_pred_labels)
    roc_auc = roc_auc_score(y_true_labels, y_scores)
    pr_auc = average_precision_score(y_true_labels, y_scores)
    tn, fp, fn, tp = confusion_matrix(y_true_labels, y_pred_labels).ravel()

    print(f"--- {model_name} on test ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print()
    print(classification_report(y_true_labels, y_pred_labels, target_names=["benign", "malicious"]))
    print("Confusion matrix (tn, fp, fn, tp):", tn, fp, fn, tp)
    print()

    return {
        "test": {
            "threshold": DECISION_THRESHOLD,
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
        },
        "test_confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        },
    }


def load_linear_svm(traditional_dir):
    vectorizer_path = os.path.join(traditional_dir, "tfidf_vectorizer_v2.joblib")
    model_path = os.path.join(traditional_dir, "linear_svm_v2.joblib")
    vectorizer = joblib.load(vectorizer_path)
    model = joblib.load(model_path)
    return vectorizer, model


def predict_linear_svm(vectorizer, model, input_texts):
    features = vectorizer.transform(input_texts)
    predictions = model.predict(features)
    scores = get_sklearn_scores(model, features)
    return predictions, scores


def load_transformer(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    return tokenizer, model


def predict_transformer(tokenizer, model, input_texts, run_device, max_len, batch_size, threshold):
    model.to(run_device)
    all_probs = []

    with torch.no_grad():
        for start_idx in range(0, len(input_texts), batch_size):
            batch_texts = input_texts[start_idx:start_idx + batch_size]
            encoded = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=max_len,
                return_tensors="pt",
            )
            encoded = {key: value.to(run_device) for key, value in encoded.items()}
            outputs = model(**encoded)
            logits = outputs.logits.squeeze(-1)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.detach().cpu().numpy().tolist())

    all_probs_array = np.array(all_probs)
    all_preds = (all_probs_array >= threshold).astype(int)
    return all_preds, all_probs_array
